In [1]:
# !pip install -q langchain langchain-openai ddgs

In [2]:
import os
import re
import json
import html
from enum import Enum

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent

In [4]:
from ddgs import DDGS
from IPython.display import HTML, display

In [5]:
API_KEY = os.environ.get('openR')
if not API_KEY:
    raise RuntimeError(
        "Environment variable 'openrouter_key' is not set. "
        "Set it (or switch to the openai/lmstudio lines below) before running this cell."
    )

llm = ChatOpenAI(temperature=0.05,
                 # model='gpt-4o-mini' #openai api users
                 # base_url='http://127.0.0.1:1234/v1' #lmstudio users
                 # api_key='no key for you' # lmstudio users
                 base_url='https://openrouter.ai/api/v1',
                 api_key=API_KEY,
                 model='deepseek/deepseek-v4-flash'
                )

In [6]:
llm.invoke('Say hi like michel scott in 10 words max.').content

"Hi, I'm Michael Scott. Let's do this!"

In [7]:
def show_gifs(gif_urls, gif_title):
    if isinstance(gif_urls, str):                   # tolerate a JSON string
        gif_urls = json.loads(gif_urls)
    unique = list(dict.fromkeys(gif_urls))          # keeps order, drops repeats
    title = html.escape(str(gif_title))             # captions can contain < or &
    if not unique:
        display(HTML(f"<h3>{title}</h3><p><i>No GIFs found.</i></p>"))
        return
    imgs = "".join(f'<img src="{url}" style="height:160px; margin:6px; border-radius:8px;">'
                   for url in unique)
    display(HTML(f"<h3>{title}</h3><div>{imgs}</div>"))

In [8]:
@tool
def search_gifs(query: str) -> list:
    '''
    search the internet for GIFs matching the given short phrase
    '''
    try:
        results = DDGS().images(f'{query} gif',     # space! otherwise we search "eye rollgif"
                                type_image='gif',
                                max_results=12
                               )
    except Exception as e:
        print(f'search failed for {query!r}: {e}')
        return []

    # just keeping th gif file
    urls = []
    for each_result in results:
        image = each_result.get('image', '')
        if '.gif' in image.lower():
            urls.append(image)
    return urls[:4]      # taking first 4 elements

In [9]:
raw = search_gifs.invoke({'query': 'eye roll'})

In [10]:
show_gifs(raw, 'eye roll')

In [11]:
raw = search_gifs.invoke({'query': 'arabic meme'})
show_gifs(raw, 'Shampoo arabic')


In [12]:
raw = search_gifs.invoke({'query': 'I am very mad at her. she ate all my lunch.'})
show_gifs(raw, 'Shampoo arabic')


In [13]:
raw = search_gifs.invoke({'query': 'Embaressed simpson'})
show_gifs(raw, 'Embaressed simpson')


# Concept 1: Planning agent

In [14]:
planner_prompt = ChatPromptTemplate.from_template("""
You are a Planning Agent for a meme reply bot.

Someone said this in a conversation:
"{input}"

You are picking the perfect reaction GIF to reply with.
First, work out the FEELING behind the message (annoyed? tired? shocked? proud?).

Now give exactly 3 reaction-GIF search phrases (2-4 words each), one for each angle:
1. the pure EMOTION       
2. a FAMOUS MEME reaction 
3. the SCENE or action 

Use words people actually type into a GIF search. Make all 3 different.
Return ONLY a JSON array of 3 strings and nothing else - no prose, no code fences.
Example: ["so done", "michael scott no", "slow clap"]
    """.strip()
)

In [15]:
planner_chain  = planner_prompt | llm

In [16]:
def parse_json_list(text: str) -> list:
    """LLMs love to wrap JSON in ``` fences or chatter. Dig the list out anyway."""
    text = text.strip()
    text = re.sub(r'^```(?:json)?|```$', '', text, flags=re.MULTILINE).strip()

    match = re.search(r'\[.*\]', text, flags=re.DOTALL)      # find the [...] block
    if match:
        try:
            items = json.loads(match.group(0))
            return [str(item).strip() for item in items if str(item).strip()]
        except json.JSONDecodeError:
            pass

    # last resort: one phrase per line, stripped of bullets / numbering / quotes
    lines = [re.sub(r'^\s*[-*\d.)]+\s*', '', line).strip(' \t,"\'')
             for line in text.splitlines()]
    return [line for line in lines if line]


def plan_searches(user_text: str) -> list:
    """Run the Planning Agent and return a clean Python list."""
    answer = planner_chain.invoke({"input": user_text}).content
    return parse_json_list(answer)[:3]

In [17]:
message = "my boss just scheduled a meeting at 5pm on a friday"
plan = plan_searches(message)
plan

['so done', 'office jim staring', 'facepalm']

# Concept 2: The Worker Agent

In [18]:
gif_agent = create_agent(
    model=llm,
    tools=[search_gifs],
    system_prompt="You are a GIF-finding worker. Use the search_gifs tool to find GIFs "
                  "for the user's phrase. Call it once, then reply with the word DONE.",
)

In [19]:
def worker_agent(phrase: str) -> list:
    """Ask the agent to find GIFs. Returns the list of urls it found."""
    result = gif_agent.invoke(
        {"messages": [{"role": "user", "content": f"Find GIFs for: {phrase}"}]},
        config={"recursion_limit": 8},        # <-- loop prevention, built in!
    )

    found = []
    for message in result["messages"]:
        if type(message).__name__ == "ToolMessage":
            content = message.content
            # our tool hands back a JSON string - be forgiving if it ever isn't
            if isinstance(content, str):
                try:
                    content = json.loads(content)
                except json.JSONDecodeError:
                    continue
            if isinstance(content, list):
                found += [url for url in content if isinstance(url, str)]
    return found

In [20]:
gifs = worker_agent("eye roll")
show_gifs(gifs, "Worker agent result")

In [21]:

result = gif_agent.invoke(
    {"messages": [{"role": "user", "content": "Find GIFs for: cat high five"}]},
    config={"recursion_limit": 8},
)

for message in result["messages"]:
    kind = type(message).__name__          # HumanMessage / AIMessage / ToolMessage
    text = str(message.content)[:70]
    print(f"{kind:14} | {text}")
    if getattr(message, "tool_calls", None):
        print(f"{'':14} |    wants to call: {message.tool_calls[0]['name']}")

HumanMessage   | Find GIFs for: cat high five
AIMessage      | 
               |    wants to call: search_gifs
ToolMessage    | ["https://c.tenor.com/pVNh6zJgK2UAAAAM/ginger-cat-funny.gif", "https:/
AIMessage      | Here are some GIFs for **cat high five**:

1. 🐱 [Ginger Cat High Five]


# Concept 3: Execution Tracking

In [22]:
class TaskStatus(Enum):
    PENDING = "⏳ Pending"
    IN_PROGRESS = "🏃 In Progress"
    COMPLETED = "✅ Completed"
    FAILED = "❌ Failed"

In [23]:
class Task:
    def __init__(self, id, description):
        self.id = id
        self.description = description
        self.status = TaskStatus.PENDING
        self.result = [] # GIF urls
    def __repr__(self):
        pass

In [24]:
class ExecutionTracker:
    def __init__(self):
        self.tasks = []
    def add_task(self, description):
        previous_tasks = len(self.tasks)
        new_task = Task(id=previous_tasks+1,
                    description=description)
        self.tasks.append(new_task)
        return new_task
    def mark_in_progress(self, new_task):
        new_task.status = TaskStatus.IN_PROGRESS
        print(f'Starting {new_task.description}')
    def mark_completed(self, task, result):
        task.status = TaskStatus.COMPLETED
        task.result = result
    def mark_failed(self, task, error):
        task.status = TaskStatus.FAILED
        print(f'Failed {task.description}: {error}')
    def print_progress(self):
        print('\nExecution Progress')
        for task in self.tasks:
            print(' ', task)
        print('-'* 60)

# Concept 4: Loop Prevention

In [25]:
@tool
def useless_search(query: str) -> str:
    '''Search for GIFs'''
    print(f'Tried to look for {query}.\nfound nothing')
    return 'No Results found, try a different phrase'

In [26]:
stubborn_agent = create_agent(model=llm, tools=[useless_search],
                              system_prompt='Keep calling the useless_search until you find the answer and Never give up'
                             )

In [27]:
try:
    stubborn_agent.invoke(
        {'messages': [{'role': 'user', 
                       'content': 'Find me an eye roll gif'
                      }]},
        config={'recursion_limit':5}
    )
except Exception as e:
    print('-'*30)
    print(f'Langchain stopped:\n{e}')

Tried to look for eye roll gif.
found nothing
Tried to look for eye roll animation.
found nothing
------------------------------
Langchain stopped:
Recursion limit of 5 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT


In [28]:
FILLER = {'a', 'an', 'the', 'gif', 'meme', 'reaction', 'of', 'to', 'and'}


class LoopGuard:
    def __init__(self, max_steps: int = 5):
        self.max_steps = max_steps
        self.history = []

    @staticmethod
    def keywords(step: str) -> set:
        """Words that actually carry meaning - "gif"/"the" overlap with everything."""
        return {word for word in step.lower().split() if word not in FILLER}

    def is_max_reached(self) -> bool:
        return len(self.history) >= self.max_steps

    def is_repeat(self, step: str) -> bool:
        """Repeat = same phrase, or 2+ meaningful words we already searched."""
        words = self.keywords(step)
        for old in self.history:
            if step.strip().lower() == old.strip().lower():
                return True
            if len(words & self.keywords(old)) >= 2:
                return True
        return False

    def remember(self, step: str):
        self.history.append(step)

# Connecting everything (finally)

In [29]:
caption_chain = ChatPromptTemplate.from_template(
    '''Someone said: "{input}"
    Reply with one short funny line(max 10 words), like a friend texting back
    '''.strip()
) | llm

In [32]:
def reply_with_meme(message):
    print(f' You said: {message}')
     #1. Convert the long sentence into few phrases
    plan = plan_searches(message)
    if not plan:
        print('Planner returned nothing - falling back to the raw message.')
        plan = [message]

    tracker = ExecutionTracker()
    guard = LoopGuard(max_steps=4)

    all_gifs = []
    for step in plan:
        if guard.is_max_reached():
            print('Max steps reached, stopping!!')
            break
        if guard.is_repeat(step):
            print(f'{step} seems to be in repeat. Skipping!!')
            continue
        guard.remember(step)
        task = tracker.add_task(step)
        tracker.mark_in_progress(task)
        try:
            gifs = worker_agent(step)
            tracker.mark_completed(task, gifs)
            all_gifs += gifs
        except Exception as e:
            tracker.mark_failed(task, str(e))

    # tracker.print_progress()
    caption = caption_chain.invoke({"input": message}).content
    show_gifs(all_gifs, caption)
    # return all_gifs

In [33]:
reply_with_meme('My client scheduled a meeting at 5pm on friday.')

 You said: My client scheduled a meeting at 5pm on friday.
Starting friday 5pm dread
Starting the office stress
Starting checking watch


In [34]:
reply_with_meme('Worked on the code for all night, now there is a problem')


 You said: Worked on the code for all night, now there is a problem
Starting exhausted programmer
Starting hide the pain harold
Starting facepalm desk


In [35]:
reply_with_meme('The dealine is comin in 10 days. only half of the task is done')

 You said: The dealine is comin in 10 days. only half of the task is done
Starting panicking
Starting this is fine
Starting freaking out


In [36]:
reply_with_meme('Tom trying to catch jerry')

 You said: Tom trying to catch jerry
Starting so frustrated
Starting tom and jerry fail
Starting chasing mouse


In [37]:
reply_with_meme('ابو مرسال')

 You said: ابو مرسال
Starting so impressed
Starting abu mersal
Starting slow nod
